# Phase 4 — Agent A6 : Synthétiseur
**YouTube Quality Analyzer** | Phase 4 — Implémentation

Ce notebook démontre l'agent A6 (`agents/a6_synthesizer.py`) :
- Agrégation pondérée : `Score_Global = 0.35×S + 0.40×D + 0.25×N`
- Attribution du `quality_label` (Faible / Moyen / Bon / Excellent)
- Génération du résumé LLM en langage naturel

> La génération de résumé est **mockée** — le calcul de score est toujours réel.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from unittest.mock import MagicMock, patch
from agents.a6_synthesizer import a6_synthesizer, _quality_label, W_SENTIMENT, W_DISCOURSE, W_NOISE

print(f"Poids PRD §4.3 : w_sentiment={W_SENTIMENT}, w_discourse={W_DISCOURSE}, w_noise={W_NOISE}")
print(f"Somme des poids : {W_SENTIMENT + W_DISCOURSE + W_NOISE}")

## 1. Calcul du Score_Global (sans LLM)

In [ ]:
# Simule les sorties de A3, A4, A5
state_good = {
    "sentiment": {"sentiment_label": "positive", "sentiment_score": 80.0, "rationale": "Mostly enthusiastic comments"},
    "discourse": {"discourse_score": 72.0, "rationale": "Several in-depth technical comments"},
    "noise": {"noise_score": 85.0, "rationale": "Low spam and off-topic content"},
}

state_bad = {
    "sentiment": {"sentiment_label": "negative", "sentiment_score": 20.0, "rationale": "Many negative comments"},
    "discourse": {"discourse_score": 15.0, "rationale": "Mostly reactions with no substance"},
    "noise": {"noise_score": 30.0, "rationale": "High spam and bot content"},
}

def compute_score(s):
    sent  = s.get("sentiment", {}).get("sentiment_score", 50.0)
    disc  = s.get("discourse",  {}).get("discourse_score",  50.0)
    noise = s.get("noise",      {}).get("noise_score",      70.0)
    return round(W_SENTIMENT * sent + W_DISCOURSE * disc + W_NOISE * noise, 2)

for label, state in [("Bon corpus", state_good), ("Mauvais corpus", state_bad)]:
    score = compute_score(state)
    qlabel = _quality_label(score)
    print(f"{label:20s} → Score_Global={score:5.1f}  Quality={qlabel}")

## 2. A6 avec LLM mocké — résumé généré

In [ ]:
mock_llm = MagicMock()
mock_llm.invoke.return_value = MagicMock(
    content="The comment section reflects strong viewer engagement with high-quality technical discourse. "
            "Sentiment is overwhelmingly positive and noise levels are low, indicating a trustworthy signal."
)

with patch("agents.a6_synthesizer.get_llm", return_value=mock_llm):
    result = a6_synthesizer(state_good)

print(f"Score_Global : {result['score_global']}")
print(f"Quality Label: {result['synthesis']['quality_label']}")
print(f"Résumé LLM   : {result['synthesis']['summary'][:120]}...")

## 3. Vérification Quality Label — tous les seuils

In [ ]:
print(f"{'Score_Global':<14} {'Quality Label'}")
print("-" * 30)
for score in [0, 10, 24.9, 25, 40, 49.9, 50, 62, 74.9, 75, 90, 100]:
    print(f"  {score:<12} {_quality_label(score)}")

print("\nPoids PRD §4.3 immuables :")
print(f"  w_sentiment + w_discourse + w_noise = {W_SENTIMENT} + {W_DISCOURSE} + {W_NOISE} = {W_SENTIMENT+W_DISCOURSE+W_NOISE}")

## Résumé A6

| Comportement | Résultat |
|---|---|
| Score_Global = 0.35×S + 0.40×D + 0.25×N | Formule PRD §4.3 immuable confirmée |
| Poids somment à 1.0 | 0.35 + 0.40 + 0.25 = 1.0 |
| Quality Label (4 niveaux) | Faible/Moyen/Bon/Excellent selon seuils |
| Résumé LLM | Généré si LLM disponible, sinon `""` |
| Erreur LLM | Score toujours calculé (fallback résumé vide) |